# TraCE-ST with real data: the 2021 Pacific Northwest heatwave

This tutorial traces causal-parent pathways backward from the 500-hPa geopotential-height anomaly associated with the June 2021 Pacific Northwest heatwave. It uses three daily anomaly fields: **Z500** (500-hPa geopotential height), **OLR** (outgoing longwave radiation, stored as `MTNLWRF`), and **TCWV** (total-column water vapor).

> **Scope.** This is a compact, analysis-ready demonstration. The paper analysis also includes Z10 and surface latent- and sensible-heat fluxes. Consequently, the pathways below are conditional on this reduced three-variable system and are not an exact reproduction of the paper results.

In [ ]:
from pathlib import Path
import copy
import warnings

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
import xarray as xr
from sklearn.exceptions import ConvergenceWarning

import trace_st as tst

warnings.filterwarnings("ignore", category=ConvergenceWarning)
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})

## 1. Load the analysis-ready fields

The distributed file has already been temporally subset, regridded to a 2° grid, and standardized consistently with the manuscript workflow. Longitudes use 0–360° east, so 240°E corresponds to 120°W. We retain the variable order below because TraCE-ST represents parent and child variables with one-based indices.

In [ ]:
candidates = [
    Path("trace_st_pnw21_processed.nc"),
    Path("Tutorials/trace_st_pnw21_processed.nc"),
]
DATA_FILE = next((path for path in candidates if path.exists()), None)
if DATA_FILE is None:
    raise FileNotFoundError("Could not find trace_st_pnw21_processed.nc")

ds = xr.open_dataset(DATA_FILE)
full_data = ds["trace_st_data"].sel(var=["Z500", "MTNLWRF", "TCWV"]).load()

VAR_LABELS = {"Z500": "Z500", "MTNLWRF": "OLR", "TCWV": "TCWV"}
var_names = [str(v) for v in full_data["var"].values]
print(f"Loaded: {DATA_FILE}")
print(f"Dimensions: {dict(full_data.sizes)}")
print(f"Period: {pd.to_datetime(full_data.time.values[0]).date()} to {pd.to_datetime(full_data.time.values[-1]).date()}")
print("Variables:", [VAR_LABELS[v] for v in var_names])

## 2. Physical state at the event peak

Z500 characterizes the mid-tropospheric circulation, OLR provides information about radiative and convective conditions, and TCWV describes column-integrated atmospheric moisture. The maps show standardized anomalies on 30 June 2021; the marker is the trajectory origin near 52.5°N, 120°W.

In [ ]:
DATE_END = pd.Timestamp("2021-06-30")
EVENT_LAT, EVENT_LON = 52.5, 240.0

event_fields = full_data.sel(time=DATE_END)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), constrained_layout=True)
for ax, name in zip(axes, var_names):
    field = event_fields.sel(var=name)
    vmax = float(np.nanpercentile(np.abs(field.values), 98))
    mesh = ax.pcolormesh(field.lon, field.lat, field, cmap="RdBu_r", vmin=-vmax, vmax=vmax, shading="auto")
    ax.scatter(EVENT_LON, EVENT_LAT, marker="*", s=100, color="black", zorder=4)
    ax.set(title=VAR_LABELS[name], xlabel="Longitude (°E)", ylabel="Latitude (°N)")
    ax.set_xlim(160, 300)
    ax.set_ylim(20, 75)
    fig.colorbar(mesh, ax=ax, shrink=0.82, label="standardized anomaly")
plt.show()

## 3. Configure the backward trajectory

The settings follow the PNW production script: a five-day learning window, 30° local box, radius-five stencil, and Elastic-Net Granger discovery. At each daily step, TraCE-ST learns local causal stencils, groups directionally coherent parents, selects a parent cluster, and moves the trajectory to the cluster's weighted displacement. The selected parent becomes the child at the next backward step.

Thirty steps provide the manuscript's 30-day horizon. For a quick test, reduce `n_steps_time` before running the next cell.

In [ ]:
params = dict(
    timeres="1d", spaceres=2.0, box_size=30.0, radius=5,
    starting_lat=EVENT_LAT, starting_lon=EVENT_LON,
    timewindow="5d", child_of_interest=1, n_steps_time=30,
    averaging_winners=True, snap_to_grid=False, montecarlo=False,
    beta_softmax=8.0, eps_dbscan=0.10, min_samples_dbscan=3,
    score_mode="mean", gamma=0.5, alpha=2.0, prefer_sign="both",
    winner_summary="mean", prob_rule="softmax", verbose=False,
    cd_method="granger",
    cd_kwargs=dict(
        lambda_a=0.5, l1_ratio=0.5, dependence_threshold=1e-8,
        max_iter=10000, fit_intercept=False, refit_ridge=False, ridge_alpha=1e-2,
    ),
)
params

In [ ]:
single = tst.trajectory.run_track(
    full_data, params, date_end=DATE_END,
    return_debug=True, return_cluster_summaries=True,
)
centers = np.asarray(single["centers"], dtype=float)
parents = np.asarray(single["parents"], dtype=int)
parent_labels = np.array([VAR_LABELS[var_names[p - 1]] for p in parents])

track_table = pd.DataFrame({
    "days_before_event": np.arange(len(centers)),
    "latitude": centers[:, 0],
    "longitude_E": centers[:, 1],
    "active_variable": parent_labels,
})
print(f"Completed {len(centers) - 1} of {params['n_steps_time']} requested steps.")
track_table.head(10)

## 4. Read the trajectory

The line is the path followed backward in time. Point colors identify the variable selected as the causal parent at each step—not the value of that variable. The timeline makes transitions between circulation, radiation/convection, and moisture parents easier to see.

In [ ]:
COLORS = {"Z500": "#3b5bdb", "OLR": "#e8590c", "TCWV": "#2b8a3e"}
fig, (ax, ax_t) = plt.subplots(2, 1, figsize=(9, 7), height_ratios=[4, 1], constrained_layout=True)
ax.plot(centers[:, 1], centers[:, 0], color="0.35", lw=1.2, zorder=1)
for label, color in COLORS.items():
    mask = parent_labels == label
    ax.scatter(centers[mask, 1], centers[mask, 0], s=35, color=color, label=label, zorder=2)
ax.scatter(EVENT_LON, EVENT_LAT, marker="*", s=150, color="black", label="event origin", zorder=3)
for i in range(0, len(centers), 5):
    ax.annotate(str(i), (centers[i, 1], centers[i, 0]), xytext=(4, 4), textcoords="offset points", fontsize=8)
ax.set(xlabel="Longitude (°E)", ylabel="Latitude (°N)", title="Backward causal-parent trajectory")
ax.grid(alpha=0.2)
ax.legend(ncol=4, fontsize=8)

y_lookup = {name: i for i, name in enumerate(COLORS)}
for i, label in enumerate(parent_labels):
    ax_t.scatter(i, y_lookup[label], color=COLORS[label], s=25)
ax_t.set(yticks=list(y_lookup.values()), yticklabels=list(y_lookup), xlabel="Days before 30 June 2021", title="Selected parent sequence")
ax_t.grid(axis="x", alpha=0.2)
plt.show()

## 5. What produces the first displacement?

For the first iteration, every significant stencil edge points from a candidate parent location to the Z500 child at the event origin. TraCE-ST converts stencil offsets into geographic displacements, separates positive and negative edges, and applies DBSCAN within each parent/sign group. The winning group is selected from these clusters; its edge-strength-weighted mean displacement is the reported $\Delta$.

The panels below show all retained stencil positions for Z500→Z500, OLR→Z500, and TCWV→Z500. Marker size represents absolute causal strength, marker shape indicates sign, and black rings mark members of the selected cluster.

In [ ]:
if not single["debug"] or single["debug"][0] is None:
    raise RuntimeError("No first-step causal candidates were retained.")

first = single["debug"][0].copy()
dlat, dlon = centers[1] - centers[0]
# The winning-edge record retains the exact global cluster selected by TraCE-ST.
chosen_cluster = int(single["winners"][0]["cluster_global"])
chosen_rows = first["cluster_global"].eq(chosen_cluster)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), sharex=True, sharey=True, constrained_layout=True)
for parent_id, (ax, name) in enumerate(zip(axes, var_names), start=1):
    sub = first[first["parent_p"] == parent_id]
    for sign, marker in [(1.0, "o"), (-1.0, "s")]:
        part = sub[sub["sign"] == sign]
        size = 25 + 180 * part["causal_value_abs"] / max(first["causal_value_abs"].max(), 1e-12)
        ax.scatter(part["delta_lon"], part["delta_lat"], s=size, marker=marker, color=COLORS[VAR_LABELS[name]], alpha=0.65)
    selected = sub[chosen_rows.loc[sub.index]]
    ax.scatter(selected["delta_lon"], selected["delta_lat"], s=180, facecolors="none", edgecolors="black", linewidths=1.4)
    ax.axhline(0, color="0.75", lw=0.8); ax.axvline(0, color="0.75", lw=0.8)
    ax.set(title=f"{VAR_LABELS[name]} → Z500", xlabel="Δ longitude (°)", ylabel="Δ latitude (°)")
    ax.set_aspect("equal")
fig.legend(handles=[Line2D([0], [0], marker="o", ls="", color="0.3", label="positive"), Line2D([0], [0], marker="s", ls="", color="0.3", label="negative")], loc="upper center", ncol=2)
plt.show()

print(f"Selected parent: {VAR_LABELS[var_names[parents[1] - 1]]}")
print(f"First displacement Δ = ({dlat:+.2f}° latitude, {dlon:+.2f}° longitude)")
single["cluster_summaries"][0]

## 6. Fixed-parameter Monte Carlo uncertainty

The deterministic track always chooses the highest-scoring cluster. With `montecarlo=True`, repeated runs sample among eligible clusters according to their score-derived probabilities while every scientific and algorithmic parameter remains fixed. This ensemble therefore represents **Monte Carlo selection uncertainty**, not sensitivity to parameter choices.

Ten members keep the tutorial reasonably lightweight. The manuscript workflow uses larger ensembles across many admissible parameter configurations; this small ensemble should not be interpreted as stable quantitative attribution.

In [ ]:
N_MEMBERS = 10
mc_params = copy.deepcopy(params)
mc_params["montecarlo"] = True

ensemble = []
for member in range(N_MEMBERS):
    try:
        out = tst.trajectory.run_track(
            full_data, mc_params, date_end=DATE_END,
            return_debug=False, return_cluster_summaries=False,
        )
        ensemble.append(out)
        print(f"member {member + 1:02d}: {len(out['centers']) - 1:2d} steps")
    except Exception as exc:
        print(f"member {member + 1:02d}: failed ({exc!r})")

print(f"Successful members: {len(ensemble)}/{N_MEMBERS}")

## 7. Ensemble pathways and parent contributions

The left panel shows the spread of sampled pathways. The right panel counts selected parent steps in three lead-time windows. Each window is normalized independently, so its bars sum to one. These fractions describe relevance within the learned reduced system; they are not fractions of physical variance explained.

In [ ]:
if not ensemble:
    raise RuntimeError("No Monte Carlo member completed successfully.")

windows = [(1, 11, "days 1–10"), (11, 21, "days 11–20"), (21, 31, "days 21–30")]
fractions = np.zeros((len(windows), len(var_names)))
for wi, (start, stop, _) in enumerate(windows):
    selected = []
    for out in ensemble:
        selected.extend(np.asarray(out["parents"], dtype=int)[start:stop])
    if selected:
        counts = np.bincount(selected, minlength=len(var_names) + 1)[1:]
        fractions[wi] = counts / counts.sum()

fig, (ax_map, ax_bar) = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
for out in ensemble:
    c = np.asarray(out["centers"], dtype=float)
    ax_map.plot(c[:, 1], c[:, 0], color="0.45", alpha=0.45, lw=1)
ax_map.scatter(EVENT_LON, EVENT_LAT, marker="*", s=140, color="black", zorder=3)
ax_map.set(xlabel="Longitude (°E)", ylabel="Latitude (°N)", title="Monte Carlo pathway spread")
ax_map.grid(alpha=0.2)

x = np.arange(len(windows)); bottom = np.zeros(len(windows))
for vi, name in enumerate(var_names):
    label = VAR_LABELS[name]
    ax_bar.bar(x, fractions[:, vi], bottom=bottom, color=COLORS[label], label=label)
    bottom += fractions[:, vi]
ax_bar.set(xticks=x, xticklabels=[w[2] for w in windows], ylim=(0, 1), ylabel="Fraction of selected parent steps", title="Integrated parent relevance")
ax_bar.legend()
plt.show()

completion = np.mean([(len(out["centers"]) - 1) == params["n_steps_time"] for out in ensemble])
print(f"Full-length completion rate among successful members: {completion:.0%}")

## Interpretation and next steps

This workflow connects local causal stencils to evolving, spatially explicit parent pathways. Three cautions are essential:

1. Causal-parent selection is conditional on the variables supplied to the model. Omitting Z10 and surface heat fluxes can alter both the chosen parents and trajectory geometry.
2. Standardized anomalies support comparison across variables, but an inferred link is not itself a physical tendency or a fraction of variance explained. Its sign must be interpreted together with each variable's convention, especially OLR.
3. The small fixed-parameter ensemble visualizes Monte Carlo selection uncertainty only. Manuscript-level robustness additionally requires larger ensembles and explicit sensitivity analysis across scientifically admissible parameter configurations.

The production and figure scripts in `Scripts_Paper/8_HW_PNW21_Runs_v2.py` and `Scripts_Paper/9_AnalysisHW_Figs45_v2.ipynb` contain the complete six-variable experiment and manuscript analysis.